# Part 6: Distance-Based Port Assignment Analysis

Section 6: Comparison of actual vs modelled port assignments using haversine distances.

**Depends on:** Part 5 (`_pipeline_state_5.pkl`)

In [ ]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 5 ─────────────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_5.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv      = _state["inv"]
links    = _state["links"]
comm_col = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT   = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS             = _state.get("SMELTERS", [])
PORTS                = _state.get("PORTS", [])
SMELTER_NAME_MAP     = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT       = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})

# ports_df is now in the state (serialised by Part 5). Reconstruct from
# PORTS as fallback in case state was generated by an older run.
ports_df = _state.get("ports_df", pd.DataFrame(PORTS))
if ports_df.empty:
    ports_df = pd.DataFrame(PORTS)

print(f"Loaded state from Part 5: {len(inv)} inv rows, {len(links)} link rows")
print(f"Ports available: {len(ports_df)}")

# ── Shared utility functions ───────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return list(matched)


In [ ]:

section_header("6. DISTANCE-BASED PORT ASSIGNMENT ANALYSIS")

mines = inv[
    (inv["CHAIN_STAGE"] == "extraction") & inv["LATITUD"].notna() &
    inv["LONGITUD"].notna() & (inv["COCHILCO_CU_2024_KMT"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"] > 0)
].copy()

# FIX: also include sx_ew facilities that have COCHILCO production matched
# directly (e.g. Zaldivar, Tres Valles) - these are pure SX-EW operations
# where no separate mine record exists in the inventory.
sxew_producers = inv[
    (inv["CHAIN_STAGE"] == "sx_ew") & inv["LATITUD"].notna() &
    inv["LONGITUD"].notna() & (inv["COCHILCO_CU_2024_KMT"].notna()) &
    (inv["COCHILCO_CU_2024_KMT"] > 0) &
    ~inv["FACILITY_NAME"].isin(mines["FACILITY_NAME"])  # not already in mines
].copy()

if len(sxew_producers) > 0:
    print(f"  Adding {len(sxew_producers)} SX-EW producers with direct production data:")
    for _, r in sxew_producers.iterrows():
        print(f"    {r['FACILITY_NAME']:<45} {r['COCHILCO_CU_2024_KMT']:.1f} kMT")
    mines = pd.concat([mines, sxew_producers], ignore_index=True)

print(f"Mines with production data: {len(mines)}")
print(f"Ports: {len(ports_df)}")

# ── Product type classification ────────────────────────────────────────────
#
# Determines whether each mine's copper output leaves Chile as concentrate,
# cathode (SX-EW or fire-refined), or blister.
#
# Priority: (1) hardcoded override, (2) comm_col on the mine record,
# (3) nearby non-mine facilities in inventory, (4) mode of PRODUCT_FORM in links.
#
# Key corrections vs. the prior run:
# - Codelco mines (Chuquicamata, El Teniente, Radomiro Tomic, etc.) are marked
#   'cathode' because their concentrate is processed onsite/nearby and the final
#   export product is refined cathode, routed via CODELCO_CATHODE_ROUTING.
# - Pure SX-EW mines (Zaldívar, Tres Valles, Antucoya, etc.) are 'cathode'.
# - Mines that ship raw concentrate for third-party smelting remain 'concentrate'.

PRODUCT_TYPE_OVERRIDE = {
    # --- Cathode: Codelco integrated operations (concentrate -> onsite smelter -> cathode) ---
    "Chuquicamata":    "cathode",  # Chuquicamata smelter -> Angamos/Mejillones
    "Radomiro Tomic":  "cathode",  # SX-EW -> Angamos (CODELCO_CATHODE_ROUTING)
    "Ministro Hales":  "cathode",  # feeds Chuquicamata smelter -> cathode
    "Gabriela Mistral": "cathode", # SX-EW (Gaby) -> Angamos
    "El Teniente":     "cathode",  # Caletones smelter -> fire-refined cathode -> San Antonio
    "Andina":          "cathode",  # feeds Las Ventanas refinery -> cathode -> Ventanas/San Antonio
    "Salvador":        "cathode",  # Potrerillos smelter + SX-EW -> Barquito
    # --- Cathode: pure SX-EW operations ---
    "Zaldívar":        "cathode",
    "Zaldivar":        "cathode",  # alternate spelling in inventory
    "El Abra":         "cathode",
    "Antucoya":        "cathode",
    "Lomas Bayas":     "cathode",
    "Mantoverde":      "cathode",  # predominantly SX-EW
    "Mantos Blancos":  "cathode",
    "Mantos de la Luna": "cathode",
    "Franke":          "cathode",
    "Tres Valles":     "cathode",
    "Michilla":        "cathode",
    "Las Luces":       "cathode",
    # --- Concentrate: mines that ship raw concentrate for third-party smelting ---
    "Escondida":       "concentrate",  # majority of output; some cathode via SX-EW
    "Collahuasi":      "concentrate",
    "Los Pelambres":   "concentrate",
    "Spence":          "concentrate",
    "Quebrada Blanca":  "concentrate",
    "Los Bronces":     "concentrate",  # -> Chagres smelter -> blister -> Ventanas
    "El Soldado":      "concentrate",  # -> Chagres smelter
    "Sierra Gorda":    "concentrate",
    "Caserones":       "concentrate",
    "Centinela":       "concentrate",
    "Candelaria":      "concentrate",
}

def classify_mine_product(mine_name):
    # 1. Hardcoded overrides (highest priority)
    for key, ptype in PRODUCT_TYPE_OVERRIDE.items():
        if key.lower() in mine_name.lower():
            return ptype

    # 2. Check comm_col on the mine record directly
    mine_rows = inv[
        inv["FACILITY_NAME"].str.contains(mine_name, case=False, na=False, regex=False) &
        inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
    ]
    if len(mine_rows) > 0 and comm_col in mine_rows.columns:
        comm_str = str(mine_rows.iloc[0].get(comm_col, "")).lower()
        if "cathode" in comm_str or "sxew" in comm_str:
            return "cathode"
        if "concentrate" in comm_str:
            return "concentrate"
        if "blister" in comm_str:
            return "blister"

    # 3. Nearby non-mine facilities in inventory
    tokens = mine_name.split()
    search_terms = [" ".join(tokens[:2])] if len(tokens) >= 2 and tokens[0] in ("El", "Los", "Las", "La") else []
    search_terms.append(tokens[0] if len(tokens[0]) >= 5 else mine_name)
    for term in search_terms:
        candidates = inv[
            inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False) &
            ~inv["FACILITY_TYPE"].str.contains("Mine|Prospect", case=False, na=False)
        ]
        if len(candidates) > 0:
            stages = set(candidates["CHAIN_STAGE"].dropna())
            if stages & {"concentration", "smelting", "processing"}: return "concentrate"
            if "sx_ew" in stages: return "cathode"

    # 4. Fallback: mode of PRODUCT_FORM in links table
    mine_links = links[links["MINE_NAME"] == mine_name]
    if len(mine_links) > 0:
        mode = mine_links["PRODUCT_FORM"].value_counts().index[0]
        return {"concentrate": "concentrate", "cathode_sxew": "cathode",
                "cathode_er": "cathode", "blister": "blister"}.get(mode, "unknown")
    return "unknown"

mines["product_type"] = mines["FACILITY_NAME"].apply(classify_mine_product)
print(f"\nMines by product type:\n{mines['product_type'].value_counts().to_string()}")

# Warn on unclassified mines so no production is silently excluded
unknown_mines = mines[mines["product_type"] == "unknown"]
if len(unknown_mines) > 0:
    unknown_prod  = unknown_mines["COCHILCO_CU_2024_KMT"].sum()
    total_prod    = mines["COCHILCO_CU_2024_KMT"].sum()
    print(f"\nWARNING: {len(unknown_mines)} mines ({unknown_prod:.1f} kMT, "
          f"{unknown_prod/total_prod*100:.1f}% of total) unclassified and excluded from port shares:")
    for _, r in unknown_mines.iterrows():
        print(f"  {r['FACILITY_NAME']:<45} {r['COCHILCO_CU_2024_KMT']:>8.1f} kMT")

# ── Vectorised distance matrix ─────────────────────────────────────────────
# Full mine x port distance matrix computed in one numpy broadcasting pass.

lat_m = np.radians(mines["LATITUD"].values)          # shape (M,)
lon_m = np.radians(mines["LONGITUD"].values)
lat_p = np.radians(ports_df["lat"].values)           # shape (P,)
lon_p = np.radians(ports_df["lon"].values)

dlat = lat_m[:, None] - lat_p[None, :]               # (M, P)
dlon = lon_m[:, None] - lon_p[None, :]
a    = (np.sin(dlat / 2) ** 2
        + np.cos(lat_m[:, None]) * np.cos(lat_p[None, :])
        * np.sin(dlon / 2) ** 2)
distances = 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

distance_df = pd.DataFrame(
    distances,
    index=mines["FACILITY_NAME"].values,
    columns=ports_df["name"].values,
)

# ── Port assignment: dedicated overrides > Codelco routing > nearest ────────

def assign_port(mine_row, dist_row):
    """Assign port using: dedicated override > Codelco routing > nearest distance."""
    mine_name = mine_row["FACILITY_NAME"].lower()
    product   = mine_row["product_type"]

    # 1. Dedicated port override (contractual)
    for key, port_name in DEDICATED_PORT.items():
        if key.lower() in mine_name and port_name in dist_row.index:
            return port_name, dist_row[port_name]

    # 2. Codelco cathode consolidation
    if product == "cathode":
        for key, port_name in CODELCO_CATHODE_ROUTING.items():
            if key.lower() in mine_name and port_name in dist_row.index:
                return port_name, dist_row[port_name]

    # 3. Nearest port by straight-line distance (default)
    return dist_row.idxmin(), dist_row.min()

assigned_ports, assigned_dists = [], []
for i, (_, mine_row) in enumerate(mines.iterrows()):
    port_name, port_dist = assign_port(mine_row, distance_df.iloc[i])
    assigned_ports.append(port_name)
    assigned_dists.append(port_dist)

mines["nearest_port"] = assigned_ports
mines["distance_km"]  = assigned_dists

print(f"\nTop 10 mines by production with assigned port:")
for _, mine in mines.nlargest(10, "COCHILCO_CU_2024_KMT").iterrows():
    mine_lower = mine["FACILITY_NAME"].lower()
    is_override = (
        any(k.lower() in mine_lower for k in DEDICATED_PORT) or
        (mine["product_type"] == "cathode" and
         any(k.lower() in mine_lower for k in CODELCO_CATHODE_ROUTING))
    )
    marker = " [override]" if is_override else ""
    print(f"  {mine['FACILITY_NAME']:<40} -> {mine['nearest_port']:<25} "
          f"({mine['distance_km']:.0f} km, {mine['COCHILCO_CU_2024_KMT']:.1f} kMT){marker}")

# ── Simulated port shares by product type ─────────────────────────────────

simulated_shares = {}
for product_type in ["concentrate", "cathode", "blister"]:
    product_mines = mines[mines["product_type"] == product_type]
    if len(product_mines) == 0: continue
    total_production = product_mines["COCHILCO_CU_2024_KMT"].sum()
    port_production  = product_mines.groupby("nearest_port")["COCHILCO_CU_2024_KMT"].sum()
    port_shares = (port_production / total_production).to_dict()
    simulated_shares[product_type] = {k: v for k, v in port_shares.items() if v >= 0.01}
    print(f"\n{product_type.upper()} (modelled):")
    for port, share in sorted(simulated_shares[product_type].items(), key=lambda x: -x[1]):
        print(f"  {port:<30} {share*100:>6.1f}%")

# ── Compare to actual Aduanas port shares ─────────────────────────────────
# Expected file: Chile_Port_Shares_Aduanas.csv
# Required columns: PORT, PRODUCT (concentrate/cathode/blister), FOB_SHARE (0-1)

port_shares_path = os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv")
if os.path.exists(port_shares_path):
    actual_shares_df = pd.read_csv(port_shares_path)
    ADUANAS_PORT_MAP = {
        "Caleta Coloso": "Coloso",
        "Puerto Angamos": "Angamos",
        "Antofagasta": "Antofagasta (ATI)",
        "Chafaral/Barquito": "Barquito",
    }
    actual_shares_df["PORT"] = actual_shares_df["PORT"].replace(ADUANAS_PORT_MAP)

    actual_shares = {}
    for product in ["concentrate", "cathode", "blister"]:
        product_data = actual_shares_df[actual_shares_df["PRODUCT"] == product]
        if len(product_data) == 0: continue
        port_share_dict = product_data.groupby("PORT")["FOB_SHARE"].sum().to_dict()
        actual_shares[product] = {k: v for k, v in port_share_dict.items() if v >= 0.01}
        print(f"\n{product.upper()} (actual, Aduanas):")
        for port, share in sorted(actual_shares[product].items(), key=lambda x: -x[1]):
            print(f"  {port:<30} {share*100:>6.1f}%")

    section_header("COMPARISON: ACTUAL vs MODELLED")
    comparison_results = []
    for product_type in ["concentrate", "cathode", "blister"]:
        if product_type not in simulated_shares: continue
        actual    = actual_shares.get(product_type, {})
        simulated = simulated_shares[product_type]
        all_ports = set(actual.keys()) | set(simulated.keys())
        print(f"\n{product_type.upper()}\n" + "-" * 75)
        for port in sorted(all_ports):
            a, s = actual.get(port, 0), simulated.get(port, 0)
            diff = s - a
            indicator = "=" if abs(diff) < 0.02 else ("^ (would gain)" if diff > 0 else "v (would lose)")
            comparison_results.append({"product": product_type, "port": port,
                                       "actual_share": a, "simulated_share": s, "difference": diff})
            print(f"  {port:<30} Actual: {a*100:5.1f}%  |  Modelled: {s*100:5.1f}%  |  D {diff*100:+6.1f}% {indicator}")

    comparison_df = pd.DataFrame(comparison_results)

    section_header("SUMMARY STATISTICS")
    for product_type in ["concentrate", "cathode", "blister"]:
        pc = comparison_df[comparison_df["product"] == product_type]
        if len(pc) == 0: continue
        print(f"\n{product_type.upper()}:")
        print(f"  Mean Absolute Difference: {pc['difference'].abs().mean()*100:.1f}%")
        print(f"  Total Volume Mismatch:    {pc['difference'].abs().sum()/2*100:.1f}%")

    comparison_df.to_csv(os.path.join(DIR_PRELIM, "Port_Distance_Comparison.csv"), index=False)
    mines[["FACILITY_NAME", "COCHILCO_CU_2024_KMT", "product_type",
           "nearest_port", "distance_km"]].to_csv(
        os.path.join(DIR_PRELIM, "Mine_Optimal_Port_Assignments.csv"), index=False)
    distance_df.to_csv(os.path.join(DIR_PRELIM, "Mine_Port_Distance_Matrix.csv"))

    # ── Visualisation ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("Port Share Comparison: Actual vs Modelled (overrides + distance)",
                 fontsize=16, fontweight="bold")
    colors = {"actual": "#FF6B6B", "simulated": "#4ECDC4"}

    for idx, product in enumerate(["concentrate", "cathode", "blister"]):
        ax = axes[idx]
        if product not in simulated_shares:
            ax.text(0.5, 0.5, f"No {product} data", ha="center", va="center", transform=ax.transAxes)
            continue
        pd_data = comparison_df[comparison_df["product"] == product].copy()
        pd_data = pd_data.sort_values("actual_share", ascending=False)
        pd_data = pd_data[(pd_data["actual_share"] >= 0.01) | (pd_data["simulated_share"] >= 0.01)].head(8)
        if len(pd_data) == 0: continue
        x     = np.arange(len(pd_data))
        width = 0.35
        ax.bar(x - width/2, pd_data["actual_share"]    * 100, width, label="Actual",   color=colors["actual"],    alpha=0.8)
        ax.bar(x + width/2, pd_data["simulated_share"] * 100, width, label="Modelled", color=colors["simulated"], alpha=0.8)
        ax.set_xlabel("Port"); ax.set_ylabel("Share (%)")
        ax.set_title(f"{product.upper()}", fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(pd_data["port"], rotation=45, ha="right", fontsize=9)
        ax.legend(); ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(DIR_PRELIM, "Port_Comparison_Chart.png"), dpi=300, bbox_inches="tight")
    print(f"\nChart saved: Port_Comparison_Chart.png")
    plt.show()

else:
    print("\nWarning: Chile_Port_Shares_Aduanas.csv not found — only modelled shares computed.")
    print(f"  Expected path: {port_shares_path}")
    print("  Required columns: PORT, PRODUCT (concentrate/cathode/blister), FOB_SHARE (0-1 decimal)")
    comparison_df = pd.DataFrame()
    mines[["FACILITY_NAME", "COCHILCO_CU_2024_KMT", "product_type",
           "nearest_port", "distance_km"]].to_csv(
        os.path.join(DIR_PRELIM, "Mine_Optimal_Port_Assignments.csv"), index=False)
    distance_df.to_csv(os.path.join(DIR_PRELIM, "Mine_Port_Distance_Matrix.csv"))
    print("Saved: Mine_Optimal_Port_Assignments.csv, Mine_Port_Distance_Matrix.csv")
